In [1]:
import numpy as np
import scipy.spatial
import pandas as pd
import sklearn.decomposition
from sklearn import preprocessing
from torch.utils.data import TensorDataset, DataLoader, Dataset, Sampler
import torch
import torch.nn as nn
from tqdm import tqdm
import copy
from Phenomix_utils import *
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from imblearn.over_sampling import RandomOverSampler
from readProfiles import *
import Phenomix


/home/sqchen/.local/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Process dataset

In [3]:
dataset='LINCS'
procProf_dir = '/home/sqchen/image/cell/dataset'
################################################
# CP Profile Type options: 'augmented' , 'normalized', 'normalized_variable_selected'
profileType='normalized_variable_selected'
profTypeAbbrev=''.join([s[0] for s in profileType.split('_')])
################################################
# filtering to compounds which have high replicates for both GE and CP datasets
# highRepOverlapEnabled=0
# 'highRepUnion','highRepOverlap',''
filter_perts='highRepUnion_and_negcon'
repCorrFilePath = 'dataset/RepCorrDF.xlsx'
filter_repCorr_params=[filter_perts,repCorrFilePath]

################################################
pertColName='PERT'
if filter_perts:
    f='filt'
else:
    f=''

mergProf_treatLevel,cp_features,l1k_features = \
read_paired_treatment_level_profiles(procProf_dir,dataset,profileType,filter_repCorr_params,1)

moa_col='Metadata_MoA'

##################################
if dataset == 'LINCS':
    mergProf_treatLevel[moa_col]=mergProf_treatLevel['Metadata_moa']
    mergProf_treatLevel.loc[mergProf_treatLevel['Metadata_moa'].isnull(),moa_col]=\
    mergProf_treatLevel.loc[mergProf_treatLevel['Metadata_moa'].isnull(),'moa'].str.lower()    
    mergProf_treatLevel['Compounds']=mergProf_treatLevel['PERT'].str[0:13]
elif dataset == 'CDRP_bio':
    mergProf_treatLevel[moa_col]=mergProf_treatLevel['Metadata_moa'].str.lower()
    mergProf_treatLevel['Compounds']=mergProf_treatLevel['PERT'].str[0:13]

l1k=mergProf_treatLevel[[pertColName,'Compounds',moa_col]+l1k_features]
cp=mergProf_treatLevel[[pertColName,'Compounds',moa_col]+cp_features]


scaler_ge = preprocessing.StandardScaler()
scaler_cp = preprocessing.StandardScaler()
l1k_scaled=l1k.copy()
l1k_scaled[l1k_features] = scaler_ge.fit_transform(l1k[l1k_features].values)
cp_scaled=cp.copy()
cp_scaled[cp_features] = scaler_cp.fit_transform(cp[cp_features].values.astype('float64'))


if 1:
    cp_scaled[cp_features] =preprocessing.MinMaxScaler(feature_range=(0, 1)).fit_transform(cp_scaled[cp_features].values)   
    l1k_scaled[l1k_features] =preprocessing.MinMaxScaler(feature_range=(0, 1)).fit_transform(l1k_scaled[l1k_features].values)           

l1k_scaled, l1k_features_gn = rename_affyprobe_to_genename(
    l1k_scaled, l1k_features, 'dataset/idmap.csv')

merged_scaled=pd.concat([cp_scaled, l1k_scaled], axis=1)
merged_scaled = merged_scaled.loc[:,~merged_scaled.columns.duplicated()]    
merged_scaled['Compounds']=merged_scaled['PERT'].str[0:13]
neg_control = merged_scaled.loc[merged_scaled['PERT'] == 'DMSO'].iloc[0, 3:]
merged_scaled.iloc[:, 3:] = merged_scaled.iloc[:, 3:]-neg_control
#################### keep MOAs with more than "nSamplesMOA" compounds in their class

nSamplesMOA=4

nSamplesforEachMOAclass=mergProf_treatLevel.groupby(['Compounds']).sample(1).groupby([moa_col]).size().\
reset_index().rename(columns={0:'size'}).sort_values(by=['size'],ascending=False).reset_index(drop=True)

nSamplesforEachMOAclass2=mergProf_treatLevel.groupby([moa_col]).size().reset_index().rename(columns={0:'size'}).sort_values(by=['size'],ascending=False).reset_index(drop=True)

listOfSelectedMoAs=nSamplesforEachMOAclass[nSamplesforEachMOAclass['size']>nSamplesMOA][moa_col].tolist()
print('If we filter to MoAs which have more than',nSamplesMOA+1,' compounds in their category, ',\
    len(listOfSelectedMoAs),' out of ',nSamplesforEachMOAclass.shape[0] ,' MoAs remain.')

multi_label_MoAs=[l for l in listOfSelectedMoAs if '|' in l]
print('There are ',len(listOfSelectedMoAs),'MoA categories, which out of them ',len(multi_label_MoAs),\
    ' have multi labels and is removed')

listOfSelectedMoAs=[ele for ele in listOfSelectedMoAs if ele not in multi_label_MoAs]

le = preprocessing.LabelEncoder()
le.fit(listOfSelectedMoAs)

filteredMOAs = merged_scaled[merged_scaled[moa_col].isin(
    listOfSelectedMoAs)].reset_index(drop=True)
filteredMOAs['Metadata_moa_num']=le.transform(filteredMOAs[moa_col].tolist())
name_list = np.array(filteredMOAs['Metadata_moa_num'].tolist())+1
align_dict = align_labels(name_list, filteredMOAs['Metadata_MoA'].tolist())
print("There are ", filteredMOAs.shape[0],"samples across different doses of ",filteredMOAs['Compounds'].unique().shape[0] ,\
    "compounds", ", for ",filteredMOAs["Metadata_MoA"].unique().shape[0], "MoAs")
data = filteredMOAs.iloc[:,3:]
data = pd.concat((data, filteredMOAs.iloc[:, 1]), axis=1)
#add novel MoA drug label in dict
align_dict[-10000] = 'Novel drug'
#avoid class 0
data['Metadata_moa_num'] += 1
data

LINCS: Replicate Level Shapes (nSamples x nFeatures): cp:  52223 , 119 ,  l1k:  27837 , 978
l1k n of rep:  3.0
cp n of rep:  5.0
CP: from  9394  to  4647
l1k: from  8369  to  2338
CP and l1k high rep union:  5845
Treatment Level Shapes (nSamples x nFeatures+metadata): (5243, 122) (4431, 980) Merged Profiles Shape: (3828, 1101)
If we filter to MoAs which have more than 5  compounds in their category,  58  out of  514  MoAs remain.
There are  58 MoA categories, which out of them  1  have multi labels and is removed
There are  1655 samples across different doses of  521 compounds , for  57 MoAs


,Cells_AreaShape_Zernike_8_4,Cytoplasm_Correlation_Overlap_ER_AGP,Cytoplasm_RadialDistribution_MeanFrac_AGP_4of4,Nuclei_Texture_InfoMeas1_AGP_20_0,Cytoplasm_Granularity_2_ER,Nuclei_AreaShape_Zernike_6_0,Cells_RadialDistribution_RadialCV_RNA_1of4,Cells_AreaShape_Zernike_9_1,Cytoplasm_Granularity_6_ER,Cytoplasm_Intensity_MassDisplacement_DNA,...,UGDH,SQOR,HEBP1,ATP11B,CD320,MLLT11,CEBPZ,CBR3,Metadata_moa_num,Compounds
0,-0.012705,0.006711,0.128779,0.002992,-0.002645,0.077156,0.044643,0.055433,-0.115735,0.075889,...,0.00348,0.039177,0.024105,0.001633,0.055267,-0.008672,0.003023,-0.010961,52,BRD-A01636364
1,0.014768,0.071332,-0.02448,0.009753,0.002817,0.005605,-0.059685,-0.03232,0.014838,-0.012217,...,0.023021,0.12178,-0.076413,0.22342,0.050085,0.017116,0.112508,-0.017954,22,BRD-A01787639
2,0.007657,0.002996,-0.004598,-0.012469,-0.001002,0.045945,-0.010979,0.002424,0.008065,-0.012167,...,0.016412,-0.0462,-0.040241,0.04971,0.033978,0.016423,0.07085,-0.078024,22,BRD-A01787639
3,-0.024297,0.010433,-0.012724,-0.026504,0.071245,-0.008773,0.028506,0.051129,0.025112,0.015034,...,-0.005874,-0.0794,-0.100882,-0.011652,-0.02204,-0.054005,0.039584,0.052418,22,BRD-A01787639
4,-0.026489,-0.003862,0.063688,0.070819,0.090301,-0.008703,0.000738,0.023959,-0.002383,-0.015945,...,-0.011621,0.036466,-0.141624,0.191508,-0.041933,-0.037688,0.057261,0.002859,22,BRD-A01787639
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1650,-0.008184,0.226836,-0.120656,-0.077723,-0.077203,0.1946,0.152372,0.10038,-0.093886,-0.043966,...,-0.037466,0.153545,0.155538,0.076639,0.0643,0.052604,0.061658,-0.063675,43,BRD-K99792991
1651,-0.031523,-0.137357,-0.018222,0.024005,-0.051941,0.021556,0.020967,0.052006,0.020538,-0.109118,...,0.001735,0.133272,0.063423,0.036976,0.009537,-0.006527,-0.008197,0.031586,33,BRD-M80207679
1652,-0.030326,-0.104173,-0.012484,0.016912,-0.066026,0.04442,0.018888,0.075916,-0.011256,-0.109393,...,0.009995,0.08567,0.028696,-0.046526,-0.004697,-0.033133,-0.143793,0.030958,33,BRD-M80207679
1653,-0.031609,-0.04654,-0.035307,-0.003849,-0.055469,0.04461,0.035605,0.052903,0.021526,-0.037808,...,0.009453,0.123864,-0.07021,0.116275,-0.033153,0.035725,0.100335,-0.006503,33,BRD-M80207679


In [ ]:
reversed_align_dict = align_labels(filteredMOAs['Metadata_MoA'].tolist(),name_list)
reversed_align_dict

{'sodium channel blocker': 52,
 'adrenergic receptor antagonist': 22,
 'calcium channel blocker': 28,
 'dopamine receptor antagonist': 33,
 'HSP inhibitor': 11,
 'CC chemokine receptor antagonist': 2,
 'histamine receptor antagonist': 36,
 'tubulin polymerization inhibitor': 56,
 'progesterone receptor agonist': 45,
 'HIV protease inhibitor': 9,
 'serotonin receptor antagonist': 51,
 'glucocorticoid receptor agonist': 35,
 'benzodiazepine receptor agonist': 27,
 'bacterial cell wall synthesis inhibitor': 26,
 'serotonin receptor agonist': 50,
 'cyclooxygenase inhibitor': 31,
 'adrenergic receptor agonist': 21,
 'acetylcholine receptor antagonist': 20,
 'bacterial DNA gyrase inhibitor': 25,
 'carbonic anhydrase inhibitor': 29,
 'monoamine oxidase inhibitor': 40,
 'androgen receptor antagonist': 23,
 'acetylcholine receptor agonist': 19,
 'angiotensin converting enzyme inhibitor': 24,
 'topoisomerase inhibitor': 55,
 'protein synthesis inhibitor': 47,
 'proteasome inhibitor': 46,
 'stero

Config

In [ ]:
LR = 0.001
EPOCH = 100
seed = 416
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
np.random.seed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
EXPLAIN = True
EXPLAIN_FOR_INDIVIDUAL_DRUG = False
NOVEL_DRUG_IDENTIFICATION = False
NOVEL_DRUG_THR = 0.5

Construct reactome knowledge network

In [ ]:
#119 CP features
cp_dim = 119
genes = data.columns[cp_dim:-2].tolist()
gene_dim = len(genes)
reactome_pathway_file = 'dataset/Reactome/ReactomePathways.txt'
reactome_pathway = pd.read_csv(reactome_pathway_file, sep='\t', names=[
                            'reactome_id', 'pathway_name', 'species'])
reactome_knowledge = get_layer_maps(genes)[0]
reactome_knowledge

layer # 0
pathways 1387
genes 9275
filtered_map (978, 0)
filtered_map (978, 0)
filtered_map (978, 0)
layer # 1
pathways 1066
genes 1399
filtered_map (1387, 0)
filtered_map (1387, 0)
filtered_map (1387, 0)
layer # 2
pathways 447
genes 1068
filtered_map (1066, 0)
filtered_map (1066, 0)
filtered_map (1066, 0)
layer # 3
pathways 147
genes 448
filtered_map (447, 0)
filtered_map (447, 0)
filtered_map (447, 0)
layer # 4
pathways 26
genes 147
filtered_map (147, 0)
filtered_map (147, 0)
filtered_map (147, 0)
layer # 5
pathways 1
genes 26
filtered_map (26, 0)
filtered_map (26, 0)
filtered_map (26, 0)


,R-HSA-110362,R-HSA-5651801,R-HSA-73930,R-HSA-111461,R-HSA-264870,R-HSA-351906,R-HSA-352238,R-HSA-111469,R-HSA-111885,R-HSA-111933,...,R-HSA-9033500,R-HSA-917977,R-HSA-975956,R-HSA-975957,R-HSA-983168,R-HSA-983170,R-HSA-983189,R-HSA-5690714,R-HSA-983695,R-HSA-936837
PSME1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
ATF1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
RHEB,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
FOXO3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
RHOA,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ATP11B,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
CD320,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
MLLT11,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
CEBPZ,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Split train and test dataset

In [ ]:
tr_data_with_compound, te_data_with_compound = train_test_split(data, test_size=0.2, random_state=1, stratify=data.iloc[:, -2])
tr_data, te_data = tr_data_with_compound.iloc[:, :-1], te_data_with_compound.iloc[:, :-1]
tr_x, tr_y = tr_data.iloc[:,:-1], tr_data.iloc[:,-1]
te_x, te_y = te_data.iloc[:, :-1], te_data.iloc[:, -1]
te_x = torch.tensor(np.array(te_x, dtype=np.float64), dtype=torch.float32)
tr_y_copy = copy.deepcopy(tr_y)

Select the classes to train with Phenomix

In [ ]:
# We run Phenomix with MEK inhibitor and PLK inhibitor as the demo
target_classes = ['MEK inhibitor','PLK inhibitor']
target_classes = [reversed_align_dict[i] for i in target_classes]
target_classes
# if you want to run all MoA classes, use the following command
# target_classes = np.unique(data['Metadata_moa_num'])

[13, 17]

In [ ]:
reactome_pathway

,reactome_id,pathway_name,species
0,R-ATH-73843,5-Phosphoribose 1-diphosphate biosynthesis,Arabidopsis thaliana
1,R-ATH-1369062,ABC transporters in lipid homeostasis,Arabidopsis thaliana
2,R-ATH-382556,ABC-family proteins mediated transport,Arabidopsis thaliana
3,R-ATH-163680,AMPK inhibits chREBP transcriptional activatio...,Arabidopsis thaliana
4,R-ATH-174143,APC/C-mediated degradation of cell cycle proteins,Arabidopsis thaliana
...,...,...,...
23436,R-XTR-191859,snRNP Assembly,Xenopus tropicalis
23437,R-XTR-379724,tRNA Aminoacylation,Xenopus tropicalis
23438,R-XTR-199992,trans-Golgi Network Vesicle Budding,Xenopus tropicalis
23439,R-XTR-140534,via Death Receptors in the presence of ligand,Xenopus tropicalis


Train

In [ ]:
crateVar = globals()
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
origin_data = copy.deepcopy(data)

for target_class in target_classes:
    model_name = f'model_{target_class}'
    tr_y = copy.deepcopy(tr_y_copy)
    tr_y[tr_y != target_class] = 0.0
    tr_y[tr_y == target_class] = 1.0
    sm1 = RandomOverSampler(sampling_strategy='not majority', random_state=5)
    tr_x_balanced, tr_y_balanced = sm1.fit_resample(
        tr_x, tr_y)
    tr_data = pd.concat((tr_x_balanced, tr_y_balanced), axis=1)
    crateVar[model_name] = Phenomix.PhenomixNet(
        reactome_knowledge, gene_dim,cp_dim).to(device)
    loss_function = nn.BCELoss()
    optimizer = torch.optim.Adam(
        globals()[model_name].parameters(),
        lr=LR
    )
    epoch = EPOCH
    moa_dataset = Phenomix.MOADataset(tr_data)
    dataloader = DataLoader(
        moa_dataset, batch_size=64, shuffle=True)
    count = 0
    for _epoch in tqdm(range(epoch)):
        for i, (x, y) in enumerate(dataloader):
            count += 1
            x, y = x.to(device), y.to(device).view(-1,1)
            output = globals()[model_name](x)
            optimizer.zero_grad()
            loss = loss_function(output, y)
            loss.backward()
            optimizer.step()
    globals()[model_name].eval().cpu()


100%|██████████| 100/100 [13:46<00:00,  8.26s/it]


Classification and Novel MoA drug identification

In [ ]:
pred_y_probs = []
pred_y = []
for target_class in target_classes:
    model_name = f'model_{target_class}'
    pred_y_prob = globals()[model_name](te_x).detach().numpy()
    pred_y_probs.append(pred_y_prob)
pred_y_indices = np.argmax(pred_y_probs, axis=0)
pred_class_probs = np.max(pred_y_probs, axis=0)
if NOVEL_DRUG_IDENTIFICATION:
    for k in range(len(pred_y_indices)):
        pred_y_indice = pred_y_indices[k][0]
        pred_class = target_classes[pred_y_indice]
        if pred_class_probs[k][0]>=NOVEL_DRUG_THR:
            pred_y.append(pred_class)
        else:
            pred_y.append(-10000)
else:
    for k in range(len(pred_y_indices)):
        pred_y_indice = pred_y_indices[k][0]
        pred_y.append(target_classes[pred_y_indice])
pred_result = {}
for i in range(len(pred_y)):
    pred_result[te_data_with_compound.iloc[i,-1]] = align_dict[pred_y[i]]
pred_result

{'BRD-A74980173': 'MEK inhibitor',
 'BRD-K46692793': 'PLK inhibitor',
 'BRD-K02900412': 'PLK inhibitor',
 'BRD-K68065987': 'MEK inhibitor',
 'BRD-K91825936': 'MEK inhibitor',
 'BRD-A24228527': 'MEK inhibitor',
 'BRD-K33882852': 'MEK inhibitor',
 'BRD-A26423207': 'MEK inhibitor',
 'BRD-K87909389': 'PLK inhibitor',
 'BRD-A19736161': 'MEK inhibitor',
 'BRD-K76617868': 'MEK inhibitor',
 'BRD-A02006392': 'MEK inhibitor',
 'BRD-K64755930': 'MEK inhibitor',
 'BRD-A38350138': 'MEK inhibitor',
 'BRD-A03506276': 'PLK inhibitor',
 'BRD-K57080016': 'MEK inhibitor',
 'BRD-K98530306': 'PLK inhibitor',
 'BRD-K16195444': 'MEK inhibitor',
 'BRD-K91308639': 'MEK inhibitor',
 'BRD-K77987382': 'PLK inhibitor',
 'BRD-K76953762': 'MEK inhibitor',
 'BRD-K13154216': 'MEK inhibitor',
 'BRD-K73088654': 'PLK inhibitor',
 'BRD-K13662825': 'MEK inhibitor',
 'BRD-K91740057': 'MEK inhibitor',
 'BRD-K70924353': 'PLK inhibitor',
 'BRD-A69951442': 'MEK inhibitor',
 'BRD-K98572433': 'MEK inhibitor',
 'BRD-K82967685': 'M

In [ ]:
np.unique(pred_y_indices)

array([0, 1])

Evaluation

In [ ]:
te_y = te_y.tolist()
target_class_te_y = []
target_class_pred_y = []
for target_class in target_classes:
    for i in range(len(te_y)):
        if te_y[i]==target_class:
            target_class_te_y.append(te_y[i])
            target_class_pred_y.append(pred_y[i])
sklearn_f1 = f1_score(target_class_te_y, target_class_pred_y, average='weighted')
print('sklearn_F1:{}\n'.format(sklearn_f1))

sklearn_F1:1.0



MoA discovery for MEK inhibitor

In [ ]:
target_class = reversed_align_dict['MEK inhibitor']
explain_model = globals()[f'model_{target_class}']
moa_explanation_result = Phenomix.moa_explanation(tr_data_with_compound, target_class, explain_model, align_dict, reactome_pathway, EXPLAIN_FOR_INDIVIDUAL_DRUG)

In [ ]:
dir(moa_explanation_result['MEK inhibitor'])

['CP_MoA_feature',
 'CP_MoA_feature_occurrence_count',
 'CP_MoA_pattern',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 'top10_genes',
 'top10_pathways']

There are 5 variables in moa_explanation_result, including CP_MoA_pattern, CP_MoA_feature, CP_MoA_feature_occurrence_count, top10_genes and top10 pathways. CP_MoA_pattern means the pattern between cell painting and MoA. CP_MoA_feature, CP_MoA_feature_occurrence_count respectively denote the feature with the highest occurrence count in the renamed top 10 most important cell painting features for MoA, and its corresponding occurrence count. top10_genes and top10_pathways respectively denote the top 10 most important genes and pathways for MoA.

In [ ]:
moa_explanation_result['MEK inhibitor'].CP_MoA_pattern,moa_explanation_result['MEK inhibitor'].CP_MoA_feature,moa_explanation_result['MEK inhibitor'].CP_MoA_feature_occurrence_count

('Nuclei', 'Nuclei_AreaShape', 5)

In [ ]:
moa_explanation_result['MEK inhibitor'].top10_genes

['ERBB2',
 'RPL39L',
 'RHOA',
 'NUP88',
 'ABL1',
 'DUSP6',
 'PSMB8',
 'PSME2',
 'DUSP4',
 'PSME1']

In [ ]:
moa_explanation_result['MEK inhibitor'].top10_pathways

['TP53 Regulates Transcription of Cell Death Genes',
 'Late Phase of HIV Life Cycle',
 'Neutrophil degranulation',
 'Neddylation',
 'Nuclear Events (kinase and transcription factor activation)',
 'MyD88-independent TLR4 cascade ',
 'Negative regulation of MAPK pathway',
 'MAP kinase activation',
 'Ub-specific processing proteases',
 'MyD88 dependent cascade initiated on endosome']

MoA explanation for PLK inhibitor

In [ ]:
target_class = reversed_align_dict['PLK inhibitor']
explain_model = globals()[f'model_{target_class}']
moa_explanation_result = Phenomix.moa_explanation(tr_data_with_compound, target_class, explain_model, align_dict, reactome_pathway, EXPLAIN_FOR_INDIVIDUAL_DRUG)

In [ ]:
moa_explanation_result['PLK inhibitor'].CP_MoA_pattern,moa_explanation_result['PLK inhibitor'].CP_MoA_feature,moa_explanation_result['PLK inhibitor'].CP_MoA_feature_occurrence_count

('RNA or Protein', 'RNA_Granularity', 2)

In [ ]:
moa_explanation_result['PLK inhibitor'].top10_genes

['EGFR',
 'RPA2',
 'CCND1',
 'POLR2I',
 'STAT1',
 'VAV3',
 'RFC5',
 'LYN',
 'PSMD9',
 'PSMD4']

In [ ]:
moa_explanation_result['PLK inhibitor'].top10_pathways

['FCERI mediated Ca+2 mobilization',
 'FCERI mediated MAPK activation',
 'Polo-like kinase mediated events',
 'Activation of the pre-replicative complex',
 'Telomere C-strand (Lagging Strand) Synthesis',
 'Interleukin-4 and Interleukin-13 signaling',
 'AURKA Activation by TPX2',
 'Neutrophil degranulation',
 'Regulation of TP53 Activity',
 'Major pathway of rRNA processing in the nucleolus and cytosol']

MoA explanation for individual drugs in MEK inhibitor

In [ ]:
target_class = reversed_align_dict['MEK inhibitor']
explain_model = globals()[f'model_{target_class}']
moa_explanation_result = Phenomix.moa_explanation(tr_data_with_compound, target_class, explain_model, align_dict, reactome_pathway, EXPLAIN_FOR_INDIVIDUAL_DRUG=True)

In [ ]:
moa_explanation_result.keys()

dict_keys(['MEK inhibitor', 'BRD-K57080016', 'BRD-K05104363', 'BRD-K64538373', 'BRD-K37687095', 'BRD-K85751432', 'BRD-K49865102', 'BRD-K03390685', 'BRD-K26667523', 'BRD-K82244583'])

In [ ]:
moa_explanation_result['BRD-K57080016'].CP_MoA_pattern,moa_explanation_result['BRD-K57080016'].CP_MoA_feature,moa_explanation_result['BRD-K57080016'].CP_MoA_feature_occurrence_count

('Nuclei', 'Nuclei_AreaShape', 5)

In [ ]:
moa_explanation_result['BRD-K57080016'].top10_genes

['ABL1',
 'SUZ12',
 'PSMD2',
 'PSMB8',
 'DUSP6',
 'MNAT1',
 'PSME2',
 'RPL39L',
 'DUSP4',
 'PSME1']

In [ ]:
moa_explanation_result['BRD-K57080016'].top10_pathways

['Late Phase of HIV Life Cycle',
 'Cyclin D associated events in G1',
 'Nuclear Events (kinase and transcription factor activation)',
 'Antigen processing: Ubiquitination & Proteasome degradation',
 'Negative regulation of MAPK pathway',
 'MAP kinase activation',
 'Neddylation',
 'MyD88 dependent cascade initiated on endosome',
 'MyD88-independent TLR4 cascade ',
 'Ub-specific processing proteases']